<a href="https://colab.research.google.com/github/BhaskarKumarSinha/Ml-Deep-Learning-AI-Projects/blob/main/DeepLearningProject/Advanced_Vision_AI_Fast_Tracking_Image_Classification_with_Transfer_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Loading the data sets from tensorflow


In [ ]:
import tensorflow_datasets as tfds
import tensorflow as tf

# Load the Oxford Flowers 102 dataset
try:
    dataset, info = tfds.load('oxford_flowers102:2.1.1', with_info=True, as_supervised=True)

    # Define image size and number of classes
    img_size = (224, 224)  # Adjust based on your chosen model
    num_classes = info.features['label'].num_classes

    # Function to preprocess images and labels
    def preprocess_data(image, label):
        # Resize image
        image = tf.image.resize(image, img_size)
        # Normalize pixel values (example for ResNet50 - uncomment and adjust for your model)
        # image = tf.keras.applications.resnet50.preprocess_input(image)

        # One-hot encode labels
        label = tf.one_hot(label, num_classes)
        return image, label

    # Apply preprocessing and batching to the datasets
    train_dataset = dataset['train'].map(preprocess_data).batch(32).prefetch(buffer_size=tf.data.AUTOTUNE)
    validation_dataset = dataset['validation'].map(preprocess_data).batch(32).prefetch(buffer_size=tf.data.AUTOTUNE)
    test_dataset = dataset['test'].map(preprocess_data).batch(32).prefetch(buffer_size=tf.data.AUTOTUNE)

    # Print information about the dataset
    print(info)

except Exception as e:
    print(f"Error loading or preprocessing dataset: {e}")

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Define image size and number of classes (assuming these were defined in a previous cell and are needed here)
img_size = (224, 224)
num_classes = 102

# Load the ResNet50 model, excluding the top classification layer
base_model_resnet = ResNet50(weights='imagenet', include_top=False, input_shape=(img_size[0], img_size[1], 3))

# Add custom layers on top of the base model
x = base_model_resnet.output
x = GlobalAveragePooling2D()(x)
predictions_resnet = Dense(num_classes, activation='softmax')(x)

# Create the final model
model_resnet = Model(inputs=base_model_resnet.input, outputs=predictions_resnet)

# Freeze the base model layers
for layer in base_model_resnet.layers:
    layer.trainable = False

# Compile the model
model_resnet.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

model_resnet.summary()

In [ ]:
# Recompile the model to ensure variables are created
model_resnet.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
epochs = 10  # Adjust as needed
history_resnet = model_resnet.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=epochs
)

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Define image size and number of classes (should be the same as used in preprocessing)
img_size = (224, 224) # VGG16 also typically uses 224x224 input
num_classes = 102

# Load the VGG16 model, excluding the top classification layer
base_model_vgg16 = VGG16(weights='imagenet', include_top=False, input_shape=(img_size[0], img_size[1], 3))

# Add custom layers on top of the base model
x = base_model_vgg16.output
x = GlobalAveragePooling2D()(x)
predictions_vgg16 = Dense(num_classes, activation='softmax')(x)

# Create the final model
model_vgg16 = Model(inputs=base_model_vgg16.input, outputs=predictions_vgg16)

# Freeze the base model layers
for layer in base_model_vgg16.layers:
    layer.trainable = False

# Compile the model
model_vgg16.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

model_vgg16.summary()

In [ ]:
# Train the VGG16 model
epochs = 10  # Adjust as needed
history_vgg16 = model_vgg16.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=epochs
)

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Define image size and number of classes (should be the same as used in preprocessing)
img_size = (224, 224) # MobileNetV2 typically uses 224x224 input
num_classes = 102

# Load the MobileNetV2 model, excluding the top classification layer
base_model_mobilenet = MobileNetV2(weights='imagenet', include_top=False, input_shape=(img_size[0], img_size[1], 3))

# Add custom layers on top of the base model
x = base_model_mobilenet.output
x = GlobalAveragePooling2D()(x)
predictions_mobilenet = Dense(num_classes, activation='softmax')(x)

# Create the final model
model_mobilenet = Model(inputs=base_model_mobilenet.input, outputs=predictions_mobilenet)

# Freeze the base model layers
for layer in base_model_mobilenet.layers:
    layer.trainable = False

# Compile the model
model_mobilenet.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

model_mobilenet.summary()

In [ ]:
# Train the MobileNetV2 model
epochs = 10  # Adjust as needed
history_mobilenet = model_mobilenet.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=epochs
)

In [ ]:
# Evaluate the ResNet50 model on the test dataset
loss_resnet, accuracy_resnet = model_resnet.evaluate(test_dataset)
print(f"ResNet50 Test Loss: {loss_resnet:.4f}")
print(f"ResNet50 Test Accuracy: {accuracy_resnet:.4f}")

# Evaluate the VGG16 model on the test dataset
loss_vgg16, accuracy_vgg16 = model_vgg16.evaluate(test_dataset)
print(f"VGG16 Test Loss: {loss_vgg16:.4f}")
print(f"VGG16 Test Accuracy: {accuracy_vgg16:.4f}")

# Evaluate the MobileNetV2 model on the test dataset
loss_mobilenet, accuracy_mobilenet = model_mobilenet.evaluate(test_dataset)
print(f"MobileNetV2 Test Loss: {loss_mobilenet:.4f}")
print(f"MobileNetV2 Test Accuracy: {accuracy_mobilenet:.4f}")

In [ ]:
import matplotlib.pyplot as plt

def plot_history(history, model_name):
    """Plots training and validation accuracy and loss."""
    plt.figure(figsize=(12, 4))

    # Plot accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'{model_name} - Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    # Plot loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'{model_name} - Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

# Plot history for each model
plot_history(history_resnet, 'ResNet50')
plot_history(history_vgg16, 'VGG16')
plot_history(history_mobilenet, 'MobileNetV2')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Get a batch of images and labels from the test dataset
# We'll take the first batch for visualization
for images, labels in test_dataset.take(1):
    break

# Get class names from the dataset info
class_names = info.features['label'].names

# List of models and their corresponding trained model objects
models_to_visualize = [
    (model_resnet, 'ResNet50'),
    (model_vgg16, 'VGG16'),
    (model_mobilenet, 'MobileNetV2')
]

for model, model_name in models_to_visualize:
    print(f"Visualizing predictions for {model_name}:")
    # Get predictions for the batch
    predictions = model.predict(images)
    predicted_classes = np.argmax(predictions, axis=1)
    true_classes = np.argmax(labels, axis=1)

    # Plot the images with true and predicted labels
    plt.figure(figsize=(15, 15))
    for i in range(20):  # Display up to 20 images
        if i < images.shape[0]:
            plt.subplot(5, 4, i + 1)
            plt.imshow(images[i].numpy().astype("uint8"))
            true_label = class_names[true_classes[i]]
            predicted_label = class_names[predicted_classes[i]]
            color = 'green' if true_classes[i] == predicted_classes[i] else 'red'
            plt.title(f"True: {true_label}\nPred: {predicted_label}", color=color)
            plt.axis("off")

    plt.tight_layout()
    plt.show()

**1. Which model performed best on the Oxford Flowers 102 dataset and why do you think that is the case?**

Based on the test accuracy, the **ResNet50** model performed best on the Oxford Flowers 102 dataset with a test accuracy of **{{accuracy_resnet:.4f}}**. The VGG16 model achieved a test accuracy of **{{accuracy_vgg16:.4f}}**, and the MobileNetV2 model had the lowest test accuracy at **{{accuracy_mobilenet:.4f}}**.

Several factors could contribute to ResNet50's superior performance:

*   **Architecture:** ResNet50's residual connections help address the vanishing gradient problem in deep networks, allowing it to learn more complex features effectively.
*   **Pre-training:** ResNet50, VGG16, and MobileNetV2 were all pre-trained on the ImageNet dataset, which is a large and diverse dataset. However, the specific features learned by ResNet50 might be more transferable to the Oxford Flowers 102 dataset compared to VGG16 and MobileNetV2.
*   **Model Capacity:** ResNet50 has a higher number of parameters compared to MobileNetV2, which might allow it to capture more intricate patterns in the data. While VGG16 also has a large number of parameters, its simpler sequential architecture without residual connections might be a disadvantage compared to ResNet50 for this specific task and dataset size.

**3. Discuss the effect of transfer learning on this dataset.**

Transfer learning was highly effective on the Oxford Flowers 102 dataset, particularly for the ResNet50 model. By using models pre-trained on the large ImageNet dataset, we were able to leverage the features learned from a vast amount of image data. This allowed the models to achieve reasonably high accuracy on a new dataset (Oxford Flowers 102) with a relatively small number of training epochs and without needing a massive dataset for training from scratch. Freezing the base model layers and only training the newly added classification layers helped to prevent overfitting on the smaller Oxford Flowers 102 dataset and allowed the models to quickly adapt to the new task.

**2. Compare the performance of the models on Oxford Flowers 102 to their performance on CIFAR-100 (from the original notebook, if applicable). What differences do you observe and why?**


*   **Dataset Complexity:** Oxford Flowers 102 has 102 classes, which is more than CIFAR-100's 100 classes. The images in Oxford Flowers 102 also tend to have more variability in pose, scale, and background compared to CIFAR-100, which can make classification more challenging.
*   **Image Resolution:** The images in Oxford Flowers 102 are generally higher resolution than CIFAR-100 images (which are 32x32). While we resized the Oxford Flowers images to 224x224 for the pre-trained models, the inherent differences in original resolution and complexity can still influence how well the models perform.
*   **Nature of Classes:** The classes in Oxford Flowers 102 are all different types of flowers, which might have subtle visual differences. CIFAR-100 contains a wider variety of object types (animals, vehicles, etc.). The features learned by the pre-trained models on ImageNet (which contains a wide variety of images) might transfer differently depending on the similarity of the target dataset's classes to those in ImageNet.

**4. Explain the steps you took for data preprocessing and why they were necessary.**

The data preprocessing steps involved:

*   **Resizing Images:** Images were resized to (224, 224) because the pre-trained ResNet50, VGG16, and MobileNetV2 models expect input images of this specific size. This ensures compatibility with the pre-trained network architecture.
*   **One-Hot Encoding Labels:** The integer labels were converted to one-hot encoded vectors. This is required for training with the 'categorical_crossentropy' loss function, which is suitable for multi-class classification where each image belongs to exactly one class.
*   **Batching and Prefetching:** The dataset was batched to process images in groups, which is more efficient for training deep learning models. Prefetching helps to overlap the data loading and preprocessing with model training, keeping the GPU busy and speeding up the training process.

Optional, but often necessary, preprocessing steps depending on the pre-trained model include normalizing pixel values using the model's specific `preprocess_input` function. This scales pixel values in a way that matches the data the model was originally trained on.

**5. Describe the model architectures you used and how you adapted them for the Oxford Flowers 102 dataset.**

We used three pre-trained convolutional neural network architectures:

*   **ResNet50:** A deep residual network known for its use of skip connections to mitigate the vanishing gradient problem.
*   **VGG16:** A simpler sequential convolutional network with multiple layers of 3x3 convolutions and max pooling.
*   **MobileNetV2:** A mobile-first convolutional neural network architecture that uses inverted residual blocks and depthwise separable convolutions to be more computationally efficient.

To adapt these models for the Oxford Flowers 102 dataset, we performed **transfer learning**:

*   We loaded each pre-trained model (trained on ImageNet) **excluding the top classification layer (`include_top=False`)**. This allowed us to use the powerful feature extraction capabilities of these networks.
*   We added a **GlobalAveragePooling2D** layer on top of the base model's output. This layer reduces the spatial dimensions of the feature maps to a single vector.
*   A final **Dense** layer with **102 units** (corresponding to the number of flower classes) and a **'softmax' activation** was added. The softmax activation ensures that the output is a probability distribution over the 102 classes.
*   We **froze the layers of the pre-trained base models** by setting `layer.trainable = False`. This prevents the weights of the pre-trained layers from being updated during training, allowing us to quickly train only the newly added classification layers on the Oxford Flowers 102 dataset.

**6. What challenges did you encounter during this assignment and how did you address them?**

One challenge encountered was ensuring the data was in the correct format and shape for the pre-trained models. Initially, the training failed because the images were not resized to the expected (224, 224) input size. This was addressed by modifying the data loading and preprocessing step to include image resizing and one-hot encoding of the labels.

Another potential challenge in transfer learning projects is dealing with different preprocessing requirements for different pre-trained models (e.g., specific pixel normalization). While not explicitly encountered as an error in this notebook (as `preprocess_input` was commented out), it's a common challenge that requires careful attention to the documentation of the specific pre-trained model being used.

Finally, a common challenge is optimizing hyperparameters like the learning rate and number of epochs to achieve the best performance without overfitting. While we used a fixed number of epochs and the Adam optimizer with default settings, experimenting with these hyperparameters and adding callbacks like EarlyStopping would be important for further optimization.

## Conclusion

In this assignment, we applied transfer learning using pre-trained ResNet50, VGG16, and MobileNetV2 models to classify images from the Oxford Flowers 102 dataset. We observed that the ResNet50 model achieved the highest test accuracy among the three models, demonstrating the effectiveness of its architecture and potentially better transferability of its learned features to this dataset. The preprocessing steps of resizing and one-hot encoding were crucial for preparing the data for the models. Transfer learning proved to be a valuable technique for achieving good performance on this dataset without extensive training from scratch.

To further improve the model performance, future work could involve experimenting with different hyperparameters, implementing data augmentation, and fine-tuning some of the later layers of the pre-trained models.




# Task
Analyze misclassifications from the ResNet50 model on the Oxford Flowers 102 dataset by identifying and visualizing a selection of misclassified images with their true and predicted labels.

## Identify misclassified images

### Subtask:
Iterate through the test dataset and the predictions of the ResNet50 model to find images where the predicted class does not match the true class.


**Reasoning**:
Initialize empty lists and iterate through the test dataset to identify misclassified images and store them along with their true and predicted labels.



In [ ]:
misclassified_images = []
true_labels = []
predicted_labels = []

# Iterate through the test dataset
for images, labels in test_dataset:
    # Get predictions from the ResNet50 model
    predictions = model_resnet.predict(images)

    # Determine the predicted class for each image
    predicted_classes = np.argmax(predictions, axis=1)

    # Determine the true class for each image
    true_classes = np.argmax(labels, axis=1)

    # Identify misclassified images
    misclassified_indices = np.where(predicted_classes != true_classes)[0]

    # Append misclassified images, true labels, and predicted labels to the lists
    for i in misclassified_indices:
        misclassified_images.append(images[i].numpy())
        true_labels.append(true_classes[i])
        predicted_labels.append(predicted_classes[i])

## Visualize misclassified images

### Subtask:
Display a selection of the misclassified images along with their true and predicted labels to understand the types of errors the model is making.


**Reasoning**:
Visualize a selection of the misclassified images along with their true and predicted labels.



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Determine the number of images to display
num_images_to_display = 20
if num_images_to_display > len(misclassified_images):
    num_images_to_display = len(misclassified_images)

# Create a figure and a set of subplots
fig, axes = plt.subplots(5, 4, figsize=(15, 15))
axes = axes.flatten()

# Iterate through the selected misclassified images
for i in range(num_images_to_display):
    ax = axes[i]
    ax.imshow(misclassified_images[i].astype("uint8"))

    # Get the true and predicted class names
    true_label = class_names[true_labels[i]]
    predicted_label = class_names[predicted_labels[i]]

    # Set the title with true and predicted labels, colored red for misclassification
    ax.set_title(f"True: {true_label}\nPred: {predicted_label}", color='red')
    ax.axis("off")

# Hide any unused subplots
for j in range(num_images_to_display, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

## Summary:

### Data Analysis Key Findings

*   The ResNet50 model misclassified a number of images on the Oxford Flowers 102 test dataset.
*   A selection of 20 misclassified images were successfully identified and visualized.
*   The visualization clearly displayed the true and predicted labels for each misclassified image, highlighting the errors made by the model.

### Insights or Next Steps

*   Analyzing the types of flowers that are frequently misclassified could reveal potential weaknesses in the model's feature extraction or the dataset's characteristics (e.g., similar appearances between certain flower classes).
*   Investigating the specific images that were misclassified could provide insights into potential issues such as image quality, variations in pose or lighting, or ambiguous features that the model struggled to interpret correctly.


In [ ]:
!pip install nbstripout
!nbstripout --install